# 🚀 OMEGA Cloud LLM GPU Runner (Google Colab Edition)
### Uncensored High-Performance Inference for Terminal AI Agents
Supports **DeepSeek V4 Pro 9B Uncensored** & **Qwen 3.8 27B Turbo Heretic** with full CUDA GPU acceleration.

In [ ]:
# @title ⚙️ 1. Setup & Install GPU Tools
# @markdown Run this cell to install Ollama with CUDA support & Cloudflare Tunnel.
import subprocess, os, sys, time, re

print("🔍 Checking GPU hardware...")
gpu = subprocess.getoutput("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader")
if "failed" in gpu.lower() or not gpu.strip():
    raise RuntimeError("❌ No GPU found! Please go to Runtime -> Change runtime type -> Select T4 GPU and restart.")
print(f"✅ Active GPU: {gpu.strip()}")

print("📦 Installing Ollama with CUDA...")
os.system("curl -fsSL https://ollama.com/install.sh | sh >/dev/null 2>&1")

print("☁️ Installing Cloudflare Tunnel...")
os.system("curl -fsSL -o /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 >/dev/null 2>&1")
os.system("chmod +x /usr/local/bin/cloudflared")
os.system("pip install -q huggingface_hub")

print("✅ Tools installed successfully! Now run Cell 2 below.")


In [ ]:
# @title 🧠 2. Launch Uncensored GPU Model & Cloudflare Tunnel
# @markdown Choose model and click Play. KEEP THIS CELL RUNNING.
MODEL_CHOICE = "DeepSeek-V4-Pro-9B-UNCENSORED (Fast, Recommended)" # @param ["DeepSeek-V4-Pro-9B-UNCENSORED (Fast, Recommended)", "Qwen3.8-27B-TURBO-Heretic-NEO-CODER"]

import os, subprocess, time, re
from huggingface_hub import hf_hub_download

os.environ["OLLAMA_HOST"] = "0.0.0.0:11434"
os.environ["OLLAMA_ORIGINS"] = "*"
os.makedirs("/content/models", exist_ok=True)

if "9B" in MODEL_CHOICE:
    REPO_ID = "xtools-at/DeepSeek-V4-Pro-Qwen3.5-9B-UNCENSORED-heretic"
    FILENAME = "DeepSeek-V4-Pro-Qwen3.5-9B-UNCENSORED-heretic-Q4_K_M.gguf"
    MODEL_NAME = "deepseek-heretic-fast"
else:
    REPO_ID = "DavidAU/Qwen3.8-27B-TURBO-Fable-Cold-Fusion-735-882-Heretic-Uncensored-NEO-CODER-MAX-MTP-GGUF"
    FILENAME = "Qwen3.8-27B-TURBO-Fable-Cold-Fusion-735-882-Heretic-Uncensored-NEO-CODER-MAX-MTP-Q3_K_M.gguf"
    MODEL_NAME = "qwen-27b-turbo"

# Download model
print(f"⬇️ Downloading {FILENAME} from Hugging Face...")
gguf_path = hf_hub_download(repo_id=REPO_ID, filename=FILENAME, local_dir="/content/models")
print(f"✅ Model weights ready: {gguf_path} ({os.path.getsize(gguf_path) / (1024**3):.2f} GB)")

# Clean restart
os.system("pkill -9 -f ollama 2>/dev/null; pkill -9 -f cloudflared 2>/dev/null")
time.sleep(2)

# Start Ollama
print("⚡ Starting Ollama CUDA server on GPU...")
ollama_proc = subprocess.Popen(["ollama", "serve"], env=os.environ)
time.sleep(5)

# Create Modelfile
print(f"🔧 Loading model {MODEL_NAME} into Ollama...")
with open("/content/Modelfile", "w") as f:
    f.write(f"FROM {gguf_path}
PARAMETER num_ctx 16384
PARAMETER temperature 0.7
")

subprocess.run(["ollama", "create", MODEL_NAME, "-f", "/content/Modelfile"], check=True)
print(f"✅ Model {MODEL_NAME} verified on GPU!")

# Start Cloudflare Tunnel
print("🌐 Starting Cloudflare Quick Tunnel...")
os.system("rm -f /content/tunnel.log")
tunnel_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://127.0.0.1:11434"],
    stdout=open("/content/tunnel.log", "w"),
    stderr=subprocess.STDOUT
)

tunnel_url = None
for _ in range(40):
    time.sleep(2)
    if os.path.exists("/content/tunnel.log"):
        with open("/content/tunnel.log", "r") as f:
            c = f.read()
            m = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", c)
            if m:
                tunnel_url = m.group(0)
                break

if tunnel_url:
    # Verify endpoint locally before announcing
    test_res = subprocess.getoutput("curl -s http://127.0.0.1:11434/api/tags")
    print("
" + "="*65)
    print("🔥 GOOGLE COLAB GPU ENDPOINT IS 100% LIVE & READY! 🔥")
    print("="*65)
    print(f"🔗 Public Tunnel URL: {tunnel_url}")
    print(f"🤖 Active Model:      {MODEL_NAME}")
    print("="*65)
    print("
📱 RUN THIS COMMAND IN TERMUX TO CONNECT ALL AGENTS:")
    print(f"link-colab {tunnel_url}")
    print("="*65)
    print("
⚠️ IMPORTANT: KEEP THIS CELL RUNNING! DO NOT STOP THIS CELL. ⚠️")
    print("="*65)
    try:
        while True:
            time.sleep(60)
    except KeyboardInterrupt:
        print("Server stopped by user.")
else:
    print("❌ Failed to get Cloudflare tunnel URL. Log contents:")
    if os.path.exists("/content/tunnel.log"):
        with open("/content/tunnel.log") as f: print(f.read())
